Hi! I am a Kaggle beginner and not fluent in English, so this notebook is written in Japanese.

Please use your browser's translation (or DeepL/ChatGPT) to read it!

Hope it helps!

# データの読み込み・ライブラリのインポート

In [1]:
import pandas as pd
import numpy as np

import lightgbm as lgb
from sklearn.model_selection import KFold
from sklearn.metrics import roc_auc_score

import kagglehub
import os

In [2]:
path = kagglehub.competition_download("playground-series-s6e10")

train = pd.read_csv(os.path.join(path, "train.csv"))
test = pd.read_csv(os.path.join(path, "test.csv"))
sample_submission = pd.read_csv(os.path.join(path, "sample_submission.csv"))

# 関数の定義

In [3]:
def train_lgb(train_df, test_df, feature_cols, target_col="satisfaction"):
    """
    指定された特徴量で LightGBM の交差検証学習を行い、
    OOF予測値（確率）、テスト予測値（確率）、学習済みモデル群を返す関数
    """
    
    # bool型をそのまま int（1/0）に変換
    y_train = train_df[target_col].astype(int).values
    
    # ループの前にカテゴリ型への一括変換
    X_train_df = train_df[feature_cols].copy()
    X_test_df = test_df[feature_cols].copy()
    
    for col in feature_cols:
        if X_train_df[col].dtype == 'object':
            X_train_df[col] = X_train_df[col].astype('category')
            X_test_df[col] = X_test_df[col].astype('category')

    X_train_df = X_train_df.reset_index(drop=True)

    cv = KFold(n_splits=5, shuffle=True, random_state=0)
    
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'boosting_type': 'gbdt',

        # 'colsample_bytree': 0.8,
        # 'subsample': 0.8,
        # 'subsample_freq': 1, # 💡 subsampleを動かす時は、1イテレーションごとにサンプリングし直すこの指定をセットで入れる
        
        'n_jobs': -1,
        'random_state': 0,
        'verbose': -1,
        'enable_categorical': True,
    }

    models = []
    oof_train = np.zeros(len(X_train_df))
    
    for fold, (train_idx, valid_idx) in enumerate(cv.split(X_train_df, y_train)):
        X_tr = X_train_df.iloc[train_idx]
        y_tr = y_train[train_idx]
        X_va = X_train_df.iloc[valid_idx]
        y_va = y_train[valid_idx]

        # Datasetの作成 (free_raw_data=True でメモリを節約)
        lgb_train = lgb.Dataset(X_tr, y_tr, free_raw_data=True)
        lgb_eval = lgb.Dataset(X_va, y_va, reference=lgb_train, free_raw_data=True)

        model = lgb.train(
            params,
            lgb_train,
            valid_sets=[lgb_eval],
            num_boost_round=1000,
            callbacks=[lgb.early_stopping(stopping_rounds=100, verbose=False)]
        )

        best_iter = model.best_iteration
        best_score = model.best_score['valid_0']['auc']
        print(f"Fold {fold+1} | Best Iteration: {best_iter:4d} | Best AUC: {best_score:.5f}")

        oof_train[valid_idx] = model.predict(X_va, num_iteration=model.best_iteration)
        
        models.append(model)

    # 全体の OOF ROC-AUC スコアを計算
    score = roc_auc_score(y_train, oof_train)
    print(f"\nOOF ROC-AUC Score: {score:.5f}")

    # すべての学習が終わった後、ループの外でテストデータを一括予測する
    y_preds = [model.predict(X_test_df, num_iteration=model.best_iteration) for model in models]
    y_preds_mean = np.mean(y_preds, axis=0)

    return oof_train, y_preds_mean, models

# 周波数エンコーディング (Frequency Encoding)

In [4]:
freq_cols = [
    "Age",
    "Flight Distance",
    "Departure Delay in Minutes",
    "Arrival Delay in Minutes",
]

# train と test を結合して全データにおける出現頻度（カウント）を計算
for col in freq_cols:
    if col in train.columns:
        # train と test の値を合わせてカウント
        full_series = pd.concat([train[col], test[col]])
        freq_map = full_series.value_counts().to_dict()

        # 新しい特徴量として追加（例: Age_freq）
        train[f"{col}_freq"] = train[col].map(freq_map)
        test[f"{col}_freq"] = test[col].map(freq_map)

print(
    f"周波数エンコーディング完了: {len(freq_cols)} 個の特徴量を生成しました。"
)

周波数エンコーディング完了: 4 個の特徴量を生成しました。


# EDA

In [5]:
# 「旅行目的（Type of Travel）」×「ロイヤリティ」のクロス特徴量
train['travel_customer_type'] = train['Type of Travel'] + "_" + train['Customer Type']
test['travel_customer_type'] = test['Type of Travel'] + "_" + test['Customer Type']

# 実行

In [6]:
%%time

feature_cols = [
    'Age',
    'Flight Distance',
    'Inflight wifi service',
    'Departure/Arrival time convenient',
    'Ease of Online booking',
    'Gate location',
    'Food and drink',
    'Online boarding',
    'Seat comfort',
    'Inflight entertainment',
    'On-board service',
    'Leg room service',
    'Baggage handling',
    'Checkin service',
    'Cleanliness',
    'Departure Delay in Minutes',
    'Arrival Delay in Minutes',
    'Gender',
    'Customer Type',
    'Type of Travel',
    'Class',
    'Age_freq',
    'Flight Distance_freq',
    'Departure Delay in Minutes_freq',
    'Arrival Delay in Minutes_freq',
    'travel_customer_type',
]


# 関数の実行
oof, preds, models = train_lgb(train, test, feature_cols)


# 特徴量重要度の可視化（表形式で出力）
feature_importances = np.mean(
    [model.feature_importance(importance_type="gain") for model in models],
    axis=0,
)

importance_df = (
    pd.DataFrame({"feature": feature_cols, "importance": feature_importances})
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

display(importance_df.head(30))

Fold 1 | Best Iteration:  338 | Best AUC: 0.95895
Fold 2 | Best Iteration:  419 | Best AUC: 0.95776
Fold 3 | Best Iteration:  617 | Best AUC: 0.96036
Fold 4 | Best Iteration:  389 | Best AUC: 0.95848
Fold 5 | Best Iteration:  642 | Best AUC: 0.96046
OOF ROC-AUC Score: 0.95920


,feature,importance
0,Online boarding,1.412516e+06
1,Inflight wifi service,3.661724e+05
2,travel_customer_type,2.064011e+05
3,Type of Travel,2.014847e+05
4,Inflight entertainment,1.364012e+05
5,Checkin service,6.492708e+04
6,Baggage handling,6.326450e+04
7,Class,5.497969e+04
8,Seat comfort,5.124236e+04
9,On-board service,4.092919e+04


CPU times: user 8min 49s, sys: 2.18 s, total: 8min 52s
Wall time: 2min 20s


# 提出

In [7]:
submission = sample_submission.copy()
submission["satisfaction"] = preds

submission.to_csv("submission.csv", index=False)
print("submission.csv を作成しました！")


# ファイルの内容・件数・欠損値のチェック
print("\n--- 先頭 5 行の確認 ---")
print(submission.head())

print("\n--- データ件数の確認 ---")
print(f"作成した件数: {len(submission)} / 元の件数: {len(sample_submission)}")

print("\n--- 欠損値（Null）の有無チェック ---")
print(f"欠損値の数: {submission['satisfaction'].isnull().sum()}")

submission.csv を作成しました！

--- 先頭 5 行の確認 ---
       id  satisfaction
0  699635      0.200614
1  699636      0.858080
2  699637      0.908034
3  699638      0.070373
4  699639      0.035061

--- データ件数の確認 ---
作成した件数: 299844 / 元の件数: 299844

--- 欠損値（Null）の有無チェック ---
欠損値の数: 0
